In [ ]:
! pip install scipy
! pip install numpy
! pip install h5py

In [1]:
import numpy as np
from scipy.sparse import diags
import h5py




class flock_plank_crank():

    def __init__(self, g: float, F):
        self.g = g
        self.D = g**2 / 2
        self.funcao_F = F
    def F(self, x: np.ndarray, t: float) -> np.ndarray:
        return self.funcao_F(x, t)
    def F_half(self, x, n,dt):

        x_half = (x[:-1] + x[1:]) / 2

        return self.F(x_half,dt*n )
    def gerar_matrix(self, n: int, dt: float, espaço: np.ndarray = np.linspace(-2,2,100)) -> tuple:
        N = len(espaço)
        dx_array = np.diff(espaço)

        if not np.allclose(dx_array, dx_array[0]):
            raise ValueError("A malha espacial precisa ser uniforme.")

        dx = dx_array[0]
        D = self.D
        F_half_n = self.F_half(espaço, n, dt)
        F_half_np1 = self.F_half(espaço, (n+1), dt)

        a_n = F_half_n / 2 + D / dx

        b_n = F_half_n / 2 - D / dx

        a_np1 = F_half_np1 / 2 + D / dx
        
        b_np1 = F_half_np1 / 2 - D / dx

        diag_A = np.zeros(N)
        diag_B = np.zeros(N)
        diag_A[0] = 1.0
        diag_B[0] = 1.0

        diag_A[-1] = 1.0
        diag_B[-1] = 1.0


        diag_A[1:-1] = (
            1
            + dt / (2 * dx) *
            (a_np1[1:] - b_np1[:-1])
        )

        diag_B[1:-1] = (
            1
            - dt / (2 * dx) *
            (a_n[1:] - b_n[:-1])
        )
        diag_baixo_A = np.zeros(N-1)
        diag_baixo_B = np.zeros(N - 1)

        diag_baixo_A[:-1] = (
            -dt / (2 * dx) * a_np1[:-1]
        )

        diag_baixo_B[:-1] = (+dt / (2 * dx) * a_n[:-1]        )

        diag_cima_A = np.zeros(N - 1)
        diag_cima_B = np.zeros(N - 1)

        diag_cima_A[1:] = (
            +dt / (2 * dx) * b_np1[1:]
        )

        diag_cima_B[1:] = (
            -dt / (2 * dx) * b_n[1:]
        )
        A = diags(
            [diag_baixo_A, diag_A, diag_cima_A],
            [-1, 0, 1],
            format="csc"
        )

        B = diags([diag_baixo_B, diag_B, diag_cima_B],[-1, 0, 1],format="csc")

        A_info = {
            "matriz_esparsa": A,
            "diag_cima": diag_cima_A,
            "diag": diag_A,
            "diag_baixo": diag_baixo_A
        }

        B_info = {
            "matriz_esparsa": B,
            "diag_cima": diag_cima_B,
            "diag": diag_B,
            "diag_baixo": diag_baixo_B
        }

        return A_info, B_info
    def evolucao(self,dt: float,p0: np.ndarray, N_max: int, espaço ):
        p_atual = np.copy(p0)

        p_completa = np.zeros((N_max, len(p0)), dtype=complex)
        p_completa[0] = p0

        for n in range(N_max - 1):

            A_info, B_info = self.gerar_matrix(
                n=n,
                dt=dt,  
                espaço=espaço
            )

            Bpn = B_info["matriz_esparsa"] @ p_atual

            p_atual = self.resolver_thomas(
                A_info["diag_cima"],
                A_info["diag_baixo"],
                A_info["diag"],
                Bpn
            )

            p_completa[n+1] = p_atual
        return p_completa
    
    def resolver_thomas(self, cima: np.ndarray, baixo: np.ndarray, meio: np.ndarray, y: np.ndarray) -> np.ndarray:
        #Ax = y
    
        J = len(meio)
        d = meio.astype(complex).copy()
        c = cima.astype(complex).copy()
        b = baixo.astype(complex).copy()
        yf = y.astype(complex).copy()

        for j in range(1, J):
            m = b[j-1] / d[j-1]
            d[j] = d[j] - m * c[j-1]
            yf[j] = yf[j] - m * yf[j-1]
        #Voltando 
        x = np.zeros(J, dtype=complex)
        x[-1] = yf[-1] / d[-1]

        for j in range(J-2, -1, -1):
            x[j] = (yf[j] - c[j] * x[j+1]) / d[j]

        return x
    

In [2]:
import numpy as np
import matplotlib.pyplot as plt

# Malha
espaço = np.linspace(-5, 5, 401)

# Parâmetros
g = 1.0
dt = 0.001
N_max = 1000

sigma0 = 0.1

# Distribuição inicial
p0 = (
    1 / np.sqrt(2*np.pi*sigma0**2)
    * np.exp(-espaço**2 / (2*sigma0**2))
)               

# Normalização numérica
p0 /= np.trapezoid(p0, espaço)  



In [3]:
def F(x, t):
    return np.zeros_like(x)

fp = flock_plank_crank(g=1.0, F=F)

In [4]:
p = fp.evolucao(
    p0=p0,
    N_max=N_max,
    dt=dt,
    espaço=espaço
)

In [ ]:
t_final = (700) * dt

sigma2 = sigma0**2 + t_final

p_exata = (
    1 / np.sqrt(2*np.pi*sigma2)
    * np.exp(-espaço**2 / (2*sigma2))
)

In [ ]:
plt.plot(espaço, p[700], label="Crank-Nicolson")
plt.plot(espaço, p_exata, "--", label="Analítica")

plt.xlabel("x")
plt.ylabel("P(x,t)")
plt.legend()
plt.show()